Predict product price from description using an LLM

In [ ]:
# imports
import os
import re
import json
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Environment Setup
# Load environment variables
load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"OpenAI API key loaded (starts with {api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")

In [ ]:
# Model Configuration
MODEL = "gpt-4.1-mini"

client = OpenAI() if api_key else None

In [ ]:
# Product Dataset
@dataclass
class Product:
    summary: str
    price: float


PRODUCTS = [
    Product("Wireless Bluetooth Earbuds, 24hr battery, noise cancelling", 49.99),
    Product("Stainless steel water bottle 32oz insulated", 29.99),
    Product("Organic cotton t-shirt men's slim fit", 24.00),
    Product("LED desk lamp with USB port, adjustable", 39.95),
    Product("Running shoes men's size 10 breathable mesh", 89.00),
    Product("Non-stick frying pan 12 inch", 34.99),
    Product("Yoga mat 6mm thick eco-friendly", 25.50),
    Product("Mechanical keyboard RGB backlit", 79.99),
    Product("Bluetooth speaker portable waterproof", 59.00),
    Product("Coffee maker 12-cup programmable", 69.99),
    Product("Backpack 25L laptop compartment", 54.99),
    Product("Smart watch fitness tracker", 129.00),
    Product("Wireless mouse ergonomic", 29.99),
    Product("Ceramic cookware set 10 piece", 119.00),
    Product("Electric toothbrush rechargeable", 45.00),
    Product("Baby stroller lightweight compact", 189.99),
    Product("Camping tent 4-person waterproof", 149.00),
    Product("Noise cancelling over-ear headphones", 199.00),
    Product("Electric kettle 1.7L stainless", 36.99),
    Product("Standing desk converter adjustable", 159.00),
]

In [ ]:
# Train/Test Split
def train_test_split(products, test_size=6, seed=42):

    import random

    rng = random.Random(seed)

    shuffled = list(products)
    rng.shuffle(shuffled)

    test = shuffled[:test_size]
    train = shuffled[test_size:]

    return train, test


train_products, test_products = train_test_split(PRODUCTS)

print(f"Train: {len(train_products)}, Test: {len(test_products)}")

In [ ]:
# Prompt Messages
def messages_for(product):

    msg = f"""
Estimate the price of this product.

Respond with ONLY the price.

Product description:
{product.summary}
"""

    return [{"role": "user", "content": msg}]

In [ ]:
# Price Parsing
# The LLM might return $49.99 or 49.99, so we clean the output.
def parse_price(value):

    if isinstance(value, (int, float)):
        return float(value)

    s = str(value).replace("$", "").replace(",", "")

    m = re.search(r"[-+]?\d*\.?\d+", s)

    return float(m.group()) if m else 0.0

In [ ]:
# Price Prediction Function
def predict_price(product, model=MODEL):

    if not client:
        return 0.0

    response = client.chat.completions.create(
        model=model,
        messages=messages_for(product),
        temperature=0
    )

    return parse_price(response.choices[0].message.content or "0")

In [ ]:
# Model Evaluation (MAE)
# Mean Absolute Error shows how accurate the predictions are.
def evaluate_mae(products):

    errors = []

    for p in products:
        prediction = predict_price(p)

        errors.append(abs(prediction - p.price))

    return sum(errors) / len(errors) if errors else 0.0


print("Predictor ready. Run the next cell to compute test MAE.")

In [ ]:
# Run Evaluation
test_mae = evaluate_mae(test_products)

print(f"Test MAE: ${test_mae:.2f}")

In [ ]:
# Optional Fine-Tuning Dataset
def make_jsonl(products):

    lines = []

    for p in products:

        messages = [
            {
                "role": "user",
                "content": f"Estimate the price of this product. Respond with the price only.\n\n{p.summary}"
            },
            {
                "role": "assistant",
                "content": f"${p.price:.2f}"
            }
        ]

        lines.append(json.dumps({"messages": messages}))

    return "\n".join(lines)


jsonl_preview = make_jsonl(train_products[:2])

print("JSONL preview:")
print(jsonl_preview)

In [ ]:
# Price Estimation Function
def estimate_price(description):

    if not description or not description.strip():
        return "Enter a product description."

    if not client:
        return "OpenAI client not initialized."

    product = Product(summary=description.strip(), price=0)

    prediction = predict_price(product)

    return f"Estimated price: ${prediction:.2f}"

In [ ]:
# Gradio UI

gr.Interface(
    fn=estimate_price,

    inputs=gr.Textbox(
        label="Product description",
        lines=3,
        placeholder="e.g. Wireless earbuds, 20hr battery..."
    ),

    outputs=gr.Textbox(label="Estimated price"),

    title="Week 6 — Product Pricer",

    description="Predict the price of a product based on its description using an LLM.",

    examples=[
        ["Wireless Bluetooth Earbuds, 24hr battery"],
        ["Stainless steel water bottle 32oz insulated"]
    ],

).launch()